# Week 7 — Hypothesis testing (simulation-first)

### Deliverable
- One completed test with effect size + CI + p-value + assumptions


In [ ]:
# Core imports (kept minimal for beginners)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Dataset URL (UCI Heart Disease - Cleveland)
UCI_URL = "https://www.archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Column names for processed.cleveland.data (14 columns commonly used in teaching)
COLS = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","num"]


In [ ]:
from scipy import stats


In [ ]:
import ssl
import io
import urllib.request # Added this import

def load_raw():
    # Create an unverified SSL context to bypass certificate verification
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    # Open the URL with the unverified context
    with urllib.request.urlopen(UCI_URL, context=ctx) as url_response:
        # Read the content and decode it
        s = url_response.read().decode('utf-8')
    
    # Use io.StringIO to make the string behave like a file for pandas.read_csv
    df_raw = pd.read_csv(io.StringIO(s), header=None, names=COLS)
    return df_raw

def coerce_types(df_raw):
    # Missing values are sometimes encoded as "?"
    df = df_raw.replace("?", np.nan).copy()

    # Convert numeric-looking columns
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Binary target: disease present if num > 0 (UCI convention)
    df["disease"] = (df["num"] > 0).astype("Int64")
    return df

df_raw = load_raw()
df = coerce_types(df_raw)

df.head()


In [ ]:
def clean_heart(df_raw):
    df = df_raw.replace("?", np.nan).copy()
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["disease"] = (df["num"] > 0).astype(int)
    return df.dropna(subset=["disease"])

df_clean = clean_heart(df_raw)


## Helper functions: permutation test + bootstrap CI


In [ ]:
rng = np.random.default_rng(0)

def permutation_test_diff_means(a, b, n_perm=5000):
    a = np.asarray(a); b = np.asarray(b)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    obs = a.mean() - b.mean()
    pooled = np.concatenate([a, b])
    n_a = len(a)
    more_extreme = 0
    for _ in range(n_perm):
        rng.shuffle(pooled)
        diff = pooled[:n_a].mean() - pooled[n_a:].mean()
        if abs(diff) >= abs(obs):
            more_extreme += 1
    p = (more_extreme + 1) / (n_perm + 1)
    return obs, p

def bootstrap_ci_diff_means(a, b, n_boot=5000, alpha=0.05):
    a = np.asarray(a); b = np.asarray(b)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    diffs = []
    for _ in range(n_boot):
        diffs.append(rng.choice(a, size=len(a), replace=True).mean()
                     - rng.choice(b, size=len(b), replace=True).mean())
    lo, hi = np.quantile(diffs, [alpha/2, 1-alpha/2])
    return (lo, hi)

def cohens_d(a, b):
    a = np.asarray(a); b = np.asarray(b)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    n1, n2 = len(a), len(b)
    s1, s2 = a.var(ddof=1), b.var(ddof=1)
    sp = ((n1-1)*s1 + (n2-1)*s2) / (n1+n2-2)
    return (a.mean() - b.mean()) / np.sqrt(sp)


## Task A — Numeric vs disease


In [ ]:
col = "thalach"  # TODO
a = df_clean.loc[df_clean["disease"]==0, col].dropna()
b = df_clean.loc[df_clean["disease"]==1, col].dropna()

obs, p_perm = permutation_test_diff_means(a, b, n_perm=5000)
ci = bootstrap_ci_diff_means(a, b, n_boot=5000)
d = cohens_d(a, b)

print("Variable:", col)
print("Mean difference (no disease - disease):", obs)
print("Permutation p-value:", p_perm)
print("Bootstrap 95% CI:", ci)
print("Cohen's d:", d)


## Task B — Categorical vs disease (chi-square)


In [ ]:
cat = "cp"  # TODO
ct = pd.crosstab(df_clean[cat], df_clean["disease"])
display(ct)

chi2, p, dof, expected = stats.chi2_contingency(ct)
print("chi2:", chi2, "p:", p)

# Cramér's V
n = ct.to_numpy().sum()
r, k = ct.shape
cramers_v = np.sqrt((chi2 / n) / (min(r-1, k-1)))
print("Cramér's V:", cramers_v)


## TODO — Assumptions & interpretation
- Assumption 1:
- Assumption 2:
- Interpretation:
- Limitation:
